## Notebook 03 - Load the clean data into PostgreSQL
**Why a database:** the SQL analysis (window functions, CTEs) needs the data in a relational database.
**Input:** `data/processed/dvf_33_2021_2025_clean.csv` (115,656 rows).
**Output:** table `sales_clean` in database `dvf_bordeaux`.

### Step 2 - Connect to PostgreSQL
I connect with SQLAlchemy to the database `dvf_bordeaux`. The password is typed at runtime (`getpass`), so it is never stored in the notebook or committed to Git.

In [3]:
import pandas as pd
from getpass import getpass
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

url = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password=getpass("PostgreSQL password: "),
    host="localhost",
    port=5432,
    database="dvf_bordeaux",
)
engine = create_engine(url)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar())

PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35229, 64-bit


### Step 3 - Load the clean table
Read `dvf_33_2021_2025_clean.csv` and fix types before loading:
- `date_mutation` becomes a real date.
- `code_postal` and `code_commune` become text with 5 digits (they are identifiers, not numbers, and the CSV turned some into decimals such as 33290.0).
Loaded into the table `sales_clean` (115,656 rows expected).

In [4]:
df = pd.read_csv("../data/processed/dvf_33_2021_2025_clean.csv", low_memory=False)

df["date_mutation"] = pd.to_datetime(df["date_mutation"]).dt.date
for c in ["code_postal", "code_commune"]:
    df[c] = df[c].astype("Int64").astype("string").str.zfill(5)

print(df.shape)
df.dtypes

(115656, 14)


id_mutation                      str
date_mutation                 object
valeur_fonciere              float64
code_postal                   string
code_commune                  string
nom_commune                      str
type_local                       str
surface_reelle_bati          float64
nombre_pieces_principales    float64
surface_terrain              float64
longitude                    float64
latitude                     float64
price_per_m2                 float64
year                           int64
dtype: object

In [5]:
df.to_sql("sales_clean", engine, if_exists="replace", index=False, chunksize=10_000)

11656

### Step 4 - Check the load
Compare the row count in PostgreSQL with the CSV (115,656) and check the sales per year.

In [6]:
pd.read_sql("SELECT year, COUNT(*) AS sales, ROUND(AVG(price_per_m2)) AS avg_price_m2 FROM sales_clean GROUP BY year ORDER BY year", engine)

,year,sales,avg_price_m2
0,2021,28097,3709.0
1,2022,27047,3895.0
2,2023,20545,3848.0
3,2024,17935,3627.0
4,2025,22032,3573.0


### Step 4 - Results
Table `sales_clean` loaded into PostgreSQL: 115,656 rows. Sales per year match the CSV: 2021: 28,097 | 2022: 27,047 | 2023: 20,545 | 2024: 17,935 | 2025: 22,032.
**Note:** the average price/m² is higher than the median every year (e.g. 3,709 vs 3,514 in 2021) because expensive sales pull the average up. The SQL analysis uses the median.